## IIT Bombay Subreddit — Data Ingestion Pipeline

This notebook ingests Reddit **posts** and **comments** from the `r/iitbombay` subreddit using **Auto Loader** for incremental processing.

**Architecture:**
- **Source**: JSON files in `/Volumes/{catalog}/{schema}/data/`
- **Posts table**: Flattened post metadata — one row per post (`post_id` as PK)
- **Comments table**: Flattened comments — one row per comment (`comment_id` as PK, `post_id` as FK → posts)
- Auto Loader handles schema inference, incremental file discovery, and exactly-once processing

**Configuration**: Use the widget inputs at the top of this notebook to set your catalog and schema.

**Relationship**: `comments.post_id` → `posts.post_id` links every comment to its parent post. Comments also have `parent_id` for thread hierarchy (`t3_` prefix = reply to post, `t1_` prefix = reply to another comment).

In [ ]:
# Widget setup - configure your catalog and schema
dbutils.widgets.text("catalog", "dbdemos_vishesh", "Catalog Name")
dbutils.widgets.text("schema", "bharat_bricks", "Schema Name")

# Get configuration from widgets
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
source_path = f"/Volumes/{catalog}/{schema}/data"
checkpoint_base = f"{source_path}/_checkpoints"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
spark.sql(f"USE SCHEMA {schema}")

print(f"Target: {catalog}.{schema}")
print(f"Source: {source_path}")
print(f"Checkpoints: {checkpoint_base}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# --- Posts Ingestion ---
posts_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", f"{checkpoint_base}/posts/schema")
    .option("pathGlobFilter", "*_posts.json")
    .load(source_path)
)

# Flatten and transform with enhanced columns
posts_transformed = posts_stream.select(
    F.col("id").alias("post_id"),
    F.col("title"),
    F.col("selftext").alias("body"),
    F.col("author"),
    F.from_unixtime("created_utc").cast("timestamp").alias("created_at"),
    F.col("permalink"),
    F.col("url"),
    F.col("score"),
    F.col("upvote_ratio"),
    F.col("num_comments"),
    F.col("link_flair_text").alias("flair"),
    F.col("author_flair_text"),
    F.col("is_self"),
    F.col("is_video"),
    F.col("is_original_content"),
    F.col("over_18").alias("is_nsfw"),
    F.col("spoiler"),
    F.col("edited"),
    F.col("locked"),
    F.col("domain"),
    F.col("num_crossposts"),
    F.col("subreddit_subscribers"),
    # Media columns
    F.col("thumbnail").alias("thumbnail_url"),
    # media_metadata is inferred as a large STRUCT (dynamic keys per media ID).
    # Convert to JSON and extract full-res source image URLs via regex on the "s":{..."u":"<url>"} pattern
    F.when(
        F.col("media_metadata").isNotNull(),
        F.regexp_extract_all(
            F.to_json(F.col("media_metadata")),
            F.lit('"s":\\{[^}]*?"u":"([^"]+)"'),
            F.lit(1)
        )
    ).alias("image_urls"),
    F.col("media.reddit_video.fallback_url").alias("video_url"),
    # Derived content type for easy filtering in dashboards
    F.when(F.col("is_video") == True, F.lit("video"))
     .when(F.col("gallery_data").isNotNull(), F.lit("gallery"))
     .when(F.col("media_metadata").isNotNull(), F.lit("image"))
     .when(
         F.col("url").rlike("(?i)\\.(jpg|jpeg|png|gif|webp)$"),
         F.lit("image")
     )
     .when(F.col("is_self") == True, F.lit("text"))
     .otherwise(F.lit("link")).alias("content_type"),
    F.col("_metadata.file_path").alias("source_file"),
)

# Write to Delta
query_posts = (
    posts_transformed.writeStream
    .option("checkpointLocation", f"{checkpoint_base}/posts/checkpoint")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.{schema}.posts")
)

query_posts.awaitTermination()
print(f"Posts ingested into {catalog}.{schema}.posts")

Posts ingested into dbdemos_vishesh.bharat_bricks.posts


In [0]:
from pyspark.sql import functions as F

# --- Comments Ingestion ---
# Comments file is a nested JSON object: {metadata: {}, processed_post_ids: [], comments: []}
# We read the whole object, then explode the 'comments' array into individual rows

comments_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", f"{checkpoint_base}/comments/schema")
    .option("pathGlobFilter", "*_comments.json")
    .load(source_path)
)

# Explode nested comments array and flatten
comments_transformed = (
    comments_stream
    .select(F.explode("comments").alias("c"), F.col("_metadata.file_path").alias("source_file"))
    .select(
        F.col("c.id").alias("comment_id"),
        F.col("c.post_id"),
        F.col("c.parent_id"),
        F.col("c.author"),
        F.col("c.body"),
        F.from_unixtime("c.created_utc").cast("timestamp").alias("created_at"),
        F.col("c.score"),
        F.col("c.depth").cast("int").alias("depth"),
        F.col("c.is_submitter"),
        F.col("c.edited"),
        F.col("c.distinguished"),
        F.col("c.permalink"),
        F.col("c.gilded").cast("int").alias("gilded"),
        F.col("c.total_awards_received").cast("int").alias("total_awards_received"),
        F.col("source_file"),
    )
)

# Write to Delta
query_comments = (
    comments_transformed.writeStream
    .option("checkpointLocation", f"{checkpoint_base}/comments/checkpoint")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.{schema}.comments")
)

query_comments.awaitTermination()
print(f"Comments ingested into {catalog}.{schema}.comments")

Comments ingested into dbdemos_vishesh.bharat_bricks.comments


In [0]:
# PK columns must be NOT NULL — set nullability first, then add constraints
for tbl, col in [("posts", "post_id"), ("comments", "comment_id")]:
    spark.sql(f"ALTER TABLE {catalog}.{schema}.{tbl} ALTER COLUMN {col} SET NOT NULL")

spark.sql(f"""
  ALTER TABLE {catalog}.{schema}.posts 
  ADD CONSTRAINT posts_pk PRIMARY KEY (post_id)
""")

spark.sql(f"""
  ALTER TABLE {catalog}.{schema}.comments 
  ADD CONSTRAINT comments_pk PRIMARY KEY (comment_id)
""")

spark.sql(f"""
  ALTER TABLE {catalog}.{schema}.comments 
  ADD CONSTRAINT comments_posts_fk FOREIGN KEY (post_id) REFERENCES {catalog}.{schema}.posts(post_id)
""")

print("PK-FK constraints added successfully")

PK-FK constraints added successfully


In [0]:
# Enable CDF on both tables for incremental downstream reads via table_changes()
for tbl in ["posts", "comments"]:
    spark.sql(f"ALTER TABLE {catalog}.{schema}.{tbl} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    print(f"Change Data Feed enabled on {catalog}.{schema}.{tbl}")

Change Data Feed enabled on dbdemos_vishesh.bharat_bricks.posts
Change Data Feed enabled on dbdemos_vishesh.bharat_bricks.comments


In [0]:
# ── Table-level comments ──
spark.sql(f"""
  COMMENT ON TABLE {catalog}.{schema}.posts IS
  'Reddit posts from r/iitbombay subreddit. Each row is one submission (text, image, video, or link). Primary key: post_id. Ingested incrementally via Auto Loader from JSON files.'
""")

spark.sql(f"""
  COMMENT ON TABLE {catalog}.{schema}.comments IS
  'Reddit comments from r/iitbombay subreddit. Each row is one comment on a post. Linked to posts via post_id (FK). Supports threaded conversations via parent_id and depth. Primary key: comment_id.'
""")

# ── Posts column comments ──
posts_column_comments = {
    "post_id":              "Unique Reddit post identifier (e.g. 1id41gh). Primary key.",
    "title":                "Title/headline of the Reddit post as written by the author.",
    "body":                 "Optional text body of the post (selftext in Reddit API). Non-empty for all text/self posts and some image/video/link posts where the author added context. Empty string when absent.",
    "author":               "Reddit username of the post author. Set to [deleted] when the account or post was removed.",
    "created_at":           "Timestamp (UTC) when the post was submitted. Converted from Unix epoch.",
    "permalink":            "Relative URL path to the post on Reddit. Prepend https://www.reddit.com to get the full URL.",
    "url":                  "Target URL of the post. For text (self) posts, points back to the Reddit post itself. For link/image/video posts, points to the external or Reddit-hosted content (i.redd.it, v.redd.it, etc.).",
    "score":                "Net vote score. Reddit floors post scores at 0 (no negatives unlike comments) and fuzzes values to deter vote manipulation.",
    "upvote_ratio":         "Fraction of all votes that are upvotes, between 0.0 and 1.0. Example: 0.95 means 95 percent upvotes.",
    "num_comments":         "Total number of comments on the post at time of data collection, including deleted and removed comments.",
    "flair":                "Post flair/category tag set by the author or moderators. Common values include Question, Other, Tech, Survey, Acads, IIT Selection, Cult, Sports, Ask me anything!, Polt. Null when untagged.",
    "author_flair_text":    "Flair tag displayed next to the author name showing affiliation. Typically a department (Elec, Mech, CS, Chem, Energy), hostel name (H5 Penthouse, H2 Wild Ones, H9 Nawaab, H4 Madhouse), student body (WnCC, Krittika, Mood Indigo), or Alum. Null for most users.",
    "is_self":              "True if this is a text/self post (body content hosted on Reddit). False for link, image, or video posts. Note: non-self posts may still have body text.",
    "is_video":             "True if the post contains a Reddit-hosted video (v.redd.it).",
    "is_original_content":  "True if the author marked this post as Original Content (OC).",
    "is_nsfw":              "True if the post is marked Not Safe For Work (over_18 flag in Reddit API).",
    "spoiler":              "True if the post is marked as containing spoilers, hiding the preview thumbnail.",
    "edited":               "String false if never edited. Contains the Unix timestamp (as string) of the last edit when the post was modified after submission.",
    "locked":               "True if moderators locked the post, preventing new comments. Typically indicates controversial or rule-breaking content.",
    "domain":               "Source domain of the post content. Common values: self.iitbombay (text posts), i.redd.it (images), v.redd.it (videos), reddit.com (crossposts). Empty string for gallery posts. May also include external domains like youtube.com and news sites.",
    "num_crossposts":       "Number of times this post was crossposted (shared) to other subreddits.",
    "subreddit_subscribers": "Number of r/iitbombay subscribers at time of data collection. May be constant within a single data snapshot.",
    "thumbnail_url":        "URL of the post thumbnail/preview image. Non-URL sentinel values: self (text post), default (no preview available), nsfw (NSFW post), spoiler (spoiler-tagged post).",
    "image_urls":           "Array of full-resolution image URLs extracted from Reddit gallery/media metadata. Null for non-image posts. Populated for posts with galleries or multi-image uploads.",
    "video_url":            "Fallback MP4 URL for Reddit-hosted videos from media.reddit_video.fallback_url. Null for non-video posts.",
    "content_type":         "Derived content classification based on is_video, gallery_data, media_metadata, URL file extension, and is_self. Possible values: text, image, gallery, video, link.",
    "source_file":          "Auto Loader metadata: full path of the source JSON file this record was ingested from. Used for lineage tracking and incremental processing.",
}

for col, comment in posts_column_comments.items():
    safe = comment.replace("'", "\\'")
    spark.sql(f"ALTER TABLE {catalog}.{schema}.posts ALTER COLUMN {col} COMMENT '{safe}'")

# ── Comments column comments ──
comments_column_comments = {
    "comment_id":           "Unique Reddit comment identifier (e.g. m8iwgz1). Primary key.",
    "post_id":              "ID of the parent post this comment belongs to. Foreign key referencing posts.post_id.",
    "parent_id":            "Reddit thing-ID of the direct parent. Prefix t3_ means reply to the post itself (top-level comment); t1_ means reply to another comment (threaded reply). Use with depth to reconstruct conversation trees.",
    "author":               "Reddit username of the commenter. Set to [deleted] when the account or comment was removed. May also contain bot/mod team accounts like iitbombay-ModTeam.",
    "body":                 "Full text content of the comment. May contain Reddit Markdown formatting (bold, links, quotes, code blocks). Set to [deleted] for user-deleted or [removed] for moderator-removed comments.",
    "created_at":           "Timestamp (UTC) when the comment was posted. Converted from Unix epoch.",
    "score":                "Net vote score (upvotes minus downvotes). Unlike posts, comment scores can go negative. Reddit fuzzes values to deter manipulation.",
    "depth":                "Nesting level in the comment thread. 0 = top-level reply to the post, 1 = reply to a top-level comment, and so on. Higher depth indicates deeper conversation threads.",
    "is_submitter":         "True if the commenter is the original poster (OP) of the parent post. Highlighted with special flair in Reddit UI.",
    "edited":               "String false if never edited. Contains the Unix timestamp (as string) of the last edit when the comment was modified after posting.",
    "distinguished":        "Set to moderator when the comment is an official mod action (e.g. content removal notices from iitbombay-ModTeam). Null for all regular user comments.",
    "permalink":            "Relative URL path to the specific comment on Reddit. Prepend https://www.reddit.com to get the full URL. Includes both the post path and comment ID.",
    "gilded":               "Number of times the comment received Reddit Gold awards.",
    "total_awards_received": "Total number of Reddit awards (Gold, Silver, community awards) received by this comment.",
    "source_file":          "Auto Loader metadata: full path of the source JSON file this record was ingested from. Used for lineage tracking and incremental processing.",
}

for col, comment in comments_column_comments.items():
    safe = comment.replace("'", "\\'")
    spark.sql(f"ALTER TABLE {catalog}.{schema}.comments ALTER COLUMN {col} COMMENT '{safe}'")

print(f"Table and column comments added for {catalog}.{schema}.posts ({len(posts_column_comments)} columns)")
print(f"Table and column comments added for {catalog}.{schema}.comments ({len(comments_column_comments)} columns)")

Table and column comments added for dbdemos_vishesh.bharat_bricks.posts (27 columns)
Table and column comments added for dbdemos_vishesh.bharat_bricks.comments (15 columns)


In [0]:
# Quick verification
posts_count = spark.table(f"{catalog}.{schema}.posts").count()
comments_count = spark.table(f"{catalog}.{schema}.comments").count()

print(f"Posts:    {posts_count:,} rows")
print(f"Comments: {comments_count:,} rows")
print(f"\nAvg comments per post: {comments_count / max(posts_count, 1):.1f}")

# Show sample
print("\n--- Sample Posts ---")
display(spark.table(f"{catalog}.{schema}.posts").select("post_id", "title", "author", "created_at", "score", "flair", "num_comments").orderBy(F.desc("score")).limit(5))

print("\n--- Sample Comments ---")
display(spark.table(f"{catalog}.{schema}.comments").select("comment_id", "post_id", "author", "body", "created_at", "score", "depth").orderBy(F.desc("score")).limit(5))

Posts:    1,339 rows
Comments: 16,790 rows

Avg comments per post: 12.5

--- Sample Posts ---


post_id,title,author,created_at,score,flair,num_comments
1id41gh,Well that pretty much sums it up.,United_Pineapple_932,2025-01-29T20:41:43.000Z,6974,Other,165
1hkhdlw,Amazing person in Techfest!!,KarmaKePakode,2024-12-23T05:54:57.000Z,5558,Other,53
1i5oxuq,The most famous dept rn 💀,relapseman,2025-01-20T12:19:36.000Z,4635,null,107
1qdbb81,IIT Bombay placements: 40 LPA + free kidnapping insurance (one-time only) thoughts?,ThalaivarThambi,2026-01-15T05:43:39.000Z,3559,Question,55
1hux93l,Illuminati Dance Crew at Techfest at IIT Bombay..!,Gracious_Heart_,2025-01-06T11:49:34.000Z,2374,null,28



--- Sample Comments ---


comment_id,post_id,author,body,created_at,score,depth
o736aky,1rd7iad,fried_liver15,It was May. My room faced Powai lake. There was a power cut. My top 1 hottest moment on campus ngl.,2026-02-24T05:28:42.000Z,117,0
nuzbc34,1pr4on4,nut_nut_november___,Tier 1 only are the people not the Institute,2025-12-20T04:12:38.000Z,108,0
o88pc16,1riuvqz,ceoofwhatthefuck,core engineering is heavily respected in iran and not seen as a liability like we do in india. their ee is one of the best in the world. those studying analog electronics would know about behzad razavi. and they do this with a fraction of budget that iits receive. they mog us hard in cs and even harder in core engineering,2026-03-02T15:20:39.000Z,94,0
nqzzazs,1p7pijl,debugyoursoul,"Chutiya log sirf naam change krne pe lage hai,",2025-11-27T04:01:30.000Z,91,0
nfkieb7,1nnhfjx,Impossible-Ice129,Next OP will discover that grass is green,2025-09-22T09:14:04.000Z,90,0
